In [1]:
import os
import re
import json
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

import joblib

In [21]:
!pip install nltk textblob gensim tensorflow scikit-learn joblib

import nltk
from textblob import TextBlob
from nltk.sentiment import SentimentIntensityAnalyzer

from gensim.models import Word2Vec

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Dropout, LSTM, Conv1D, GlobalMaxPooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

nltk.download("vader_lexicon")
sia = SentimentIntensityAnalyzer()

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 1.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.1/64.1 kB 1.3 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/mac/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [29]:
# Change this path if your CSV is in a data/ folder
DATA_PATH = "IMDB Dataset.csv"

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(50000, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [30]:
def clean_review(x):
    if not isinstance(x, str):
        return ""
    x = x.lower()
    x = re.sub(r"<br\s*/?>", " ", x)
    x = re.sub(r"[^a-z0-9\s']", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

df["review_clean"] = df["review"].astype(str).apply(clean_review)
df["y"] = df["sentiment"].str.lower().map({"positive": 1, "negative": 0})

print(df["y"].value_counts())
df[["review_clean", "y"]].head()

y
1    25000
0    25000
Name: count, dtype: int64


,review_clean,y
0,one of the other reviewers has mentioned that ...,1
1,a wonderful little production the filming tech...,1
2,i thought this was a wonderful way to spend ti...,1
3,basically there's a family where a little boy ...,0
4,petter mattei's love in the time of money is a...,1


In [31]:
train_df, test_df = train_test_split(
    df[["review_clean", "y"]],
    test_size=0.2,
    random_state=SEED,
    stratify=df["y"]
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=SEED,
    stratify=train_df["y"]
)

print("train:", train_df.shape)
print("val:  ", val_df.shape)
print("test: ", test_df.shape)

train: (36000, 2)
val:   (4000, 2)
test:  (10000, 2)


In [32]:
os.makedirs("assignment2", exist_ok=True)

def save_results_and_readme(model_dir, metrics, readme_text):
    os.makedirs(model_dir, exist_ok=True)

    with open(os.path.join(model_dir, "results.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    with open(os.path.join(model_dir, "README.md"), "w") as f:
        f.write(readme_text)

def evaluate_predictions(y_true, y_pred, y_prob):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred)),
        "recall": float(recall_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred)),
        "roc_auc": float(roc_auc_score(y_true, y_prob))
    }

In [33]:
#model 1: tuning MLP with VADER + TextBlob features (larger architecture and cleaner preprocessing)
def get_vader_feats(text):
    s = sia.polarity_scores(text)
    return s["neg"], s["neu"], s["pos"], s["compound"]

def get_textblob_feats(text):
    t = TextBlob(text).sentiment
    return float(t.polarity), float(t.subjectivity)

def build_lexicon_features(text_series):
    vader_feats = text_series.apply(get_vader_feats)
    textblob_feats = text_series.apply(get_textblob_feats)

    feats = pd.DataFrame({
        "tb_pol": textblob_feats.apply(lambda x: x[0]),
        "tb_subj": textblob_feats.apply(lambda x: x[1]),
        "vader_neg": vader_feats.apply(lambda x: x[0]),
        "vader_neu": vader_feats.apply(lambda x: x[1]),
        "vader_pos": vader_feats.apply(lambda x: x[2]),
        "vader_comp": vader_feats.apply(lambda x: x[3]),
    })
    return feats

X_train_lex = build_lexicon_features(train_df["review_clean"])
X_val_lex = build_lexicon_features(val_df["review_clean"])
X_test_lex = build_lexicon_features(test_df["review_clean"])

y_train = train_df["y"].values
y_val = val_df["y"].values
y_test = test_df["y"].values

X_train_lex.head()

,tb_pol,tb_subj,vader_neg,vader_neu,vader_pos,vader_comp
24839,0.088360,0.497431,0.156,0.686,0.158,0.1906
33890,0.253333,0.600370,0.107,0.696,0.198,0.9241
4692,0.083834,0.476230,0.113,0.752,0.135,0.9505
34202,-0.117708,0.565972,0.128,0.777,0.095,-0.8035
29985,0.058664,0.491586,0.148,0.731,0.121,-0.9688


In [34]:
mlp_lex = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    batch_size=256,
    learning_rate_init=0.001,
    max_iter=200,
    random_state=SEED,
    verbose=False
)

model_lex = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", mlp_lex)
])

model_lex.fit(X_train_lex, y_train)

pred_lex = model_lex.predict(X_test_lex)
prob_lex = model_lex.predict_proba(X_test_lex)[:, 1]

metrics_lex = evaluate_predictions(y_test, pred_lex, prob_lex)
metrics_lex

{'accuracy': 0.774,
 'precision': 0.7742193755004003,
 'recall': 0.7736,
 'f1': 0.7739095638255302,
 'roc_auc': 0.8596550399999999}

In [28]:
print("Tuned lexicon MLP")
print(metrics_lex)
print("\nClassification report:\n")
print(classification_report(y_test, pred_lex, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, pred_lex))

model_dir = "assignment2/mlp_lexicon_tuned"
os.makedirs(model_dir, exist_ok=True)

joblib.dump(model_lex, os.path.join(model_dir, "model.joblib"))

readme_lex = """# Tuned MLP with Lexicon Features

## Model architecture
- Input features: 6 sentiment-based features
  - TextBlob polarity
  - TextBlob subjectivity
  - VADER neg / neu / pos / compound
- StandardScaler
- MLPClassifier with hidden layers (64, 32)
- ReLU activation
- Adam optimizer

## Techniques applied
- text cleaning
- lexicon feature engineering
- scaling
- hyperparameter tuning (larger hidden layers and more iterations than Assignment 1)
- evaluation with accuracy, precision, recall, F1, and ROC-AUC
"""

save_results_and_readme(model_dir, metrics_lex, readme_lex)

Tuned lexicon MLP
{'accuracy': 0.774, 'precision': 0.7742193755004003, 'recall': 0.7736, 'f1': 0.7739095638255302, 'roc_auc': 0.8596550399999999}

Classification report:

              precision    recall  f1-score   support

           0     0.7738    0.7744    0.7741      5000
           1     0.7742    0.7736    0.7739      5000

    accuracy                         0.7740     10000
   macro avg     0.7740    0.7740    0.7740     10000
weighted avg     0.7740    0.7740    0.7740     10000

Confusion matrix:
 [[3872 1128]
 [1132 3868]]


FileNotFoundError: [Errno 2] No such file or directory: 'assignment2/mlp_lexicon_tuned/model.joblib'

In [35]:
#Model 2: LSTM (tokenized review sequences instead of only handcrafted sentiment scores)
MAX_WORDS = 10000
MAX_LEN = 200
EMBED_DIM = 100

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["review_clean"])

X_train_seq = tokenizer.texts_to_sequences(train_df["review_clean"])
X_val_seq = tokenizer.texts_to_sequences(val_df["review_clean"])
X_test_seq = tokenizer.texts_to_sequences(test_df["review_clean"])

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post", truncating="post")

print(X_train_pad.shape, X_val_pad.shape, X_test_pad.shape)

(36000, 200) (4000, 200) (10000, 200)


In [36]:
lstm_model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=EMBED_DIM, input_length=MAX_LEN),
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    Dense(64, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history_lstm = lstm_model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=5,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/5


/Users/mac/.pyenv/versions/3.11.9/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


282/282 ━━━━━━━━━━━━━━━━━━━━ 61s 208ms/step - accuracy: 0.5553 - loss: 0.6835 - val_accuracy: 0.6185 - val_loss: 0.6682
Epoch 2/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 57s 202ms/step - accuracy: 0.6438 - loss: 0.6477 - val_accuracy: 0.6438 - val_loss: 0.6464
Epoch 3/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 59s 208ms/step - accuracy: 0.6033 - loss: 0.6512 - val_accuracy: 0.5985 - val_loss: 0.6349
Epoch 4/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 58s 206ms/step - accuracy: 0.7426 - loss: 0.5444 - val_accuracy: 0.7868 - val_loss: 0.5151
Epoch 5/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 59s 211ms/step - accuracy: 0.7787 - loss: 0.5004 - val_accuracy: 0.6225 - val_loss: 0.6531


In [37]:
prob_lstm = lstm_model.predict(X_test_pad).ravel()
pred_lstm = (prob_lstm >= 0.5).astype(int)

metrics_lstm = evaluate_predictions(y_test, pred_lstm, prob_lstm)
metrics_lstm

313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step


{'accuracy': 0.7887,
 'precision': 0.7817684950224478,
 'recall': 0.801,
 'f1': 0.791267410846587,
 'roc_auc': 0.8342927200000001}

In [40]:
print("LSTM")
print(metrics_lstm)
print("\nClassification report:\n")
print(classification_report(y_test, pred_lstm, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, pred_lstm))

model_dir = "assignment2/lstm"
os.makedirs(model_dir, exist_ok=True)

lstm_model.save(os.path.join(model_dir, "model.keras"))

readme_lstm = """# LSTM for IMDB Sentiment Classification

## Model architecture
- Input: tokenized and padded review sequences
- Embedding layer (trainable)
- LSTM layer with 64 units
- Dense layer with ReLU activation
- Dropout regularization
- Output sigmoid layer

## Techniques applied
- text cleaning
- tokenization and padding
- recurrent neural network (LSTM)
- dropout regularization
- early stopping
- evaluation with accuracy, precision, recall, F1, and ROC-AUC
"""

save_results_and_readme(model_dir, metrics_lstm, readme_lstm)

LSTM
{'accuracy': 0.7887, 'precision': 0.7817684950224478, 'recall': 0.801, 'f1': 0.791267410846587, 'roc_auc': 0.8342927200000001}

Classification report:

              precision    recall  f1-score   support

           0     0.7960    0.7764    0.7861      5000
           1     0.7818    0.8010    0.7913      5000

    accuracy                         0.7887     10000
   macro avg     0.7889    0.7887    0.7887     10000
weighted avg     0.7889    0.7887    0.7887     10000

Confusion matrix:
 [[3882 1118]
 [ 995 4005]]


In [41]:
# Model 3: 1D CNN with trainable embedding layer

cnn_model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=EMBED_DIM, input_length=MAX_LEN),
    Conv1D(filters=128, kernel_size=5, activation="relu"),
    GlobalMaxPooling1D(),
    Dense(64, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

cnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_cnn = cnn_model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=3,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/3


/Users/mac/.pyenv/versions/3.11.9/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


282/282 ━━━━━━━━━━━━━━━━━━━━ 14s 47ms/step - accuracy: 0.7414 - loss: 0.4983 - val_accuracy: 0.8690 - val_loss: 0.3190
Epoch 2/3
282/282 ━━━━━━━━━━━━━━━━━━━━ 12s 44ms/step - accuracy: 0.8952 - loss: 0.2691 - val_accuracy: 0.8805 - val_loss: 0.2911
Epoch 3/3
282/282 ━━━━━━━━━━━━━━━━━━━━ 13s 45ms/step - accuracy: 0.9491 - loss: 0.1577 - val_accuracy: 0.8865 - val_loss: 0.2982


In [43]:
prob_cnn = cnn_model.predict(X_test_pad).ravel()
pred_cnn = (prob_cnn >= 0.5).astype(int)

metrics_cnn = evaluate_predictions(y_test, pred_cnn, prob_cnn)
metrics_cnn

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


{'accuracy': 0.8762,
 'precision': 0.9034749034749034,
 'recall': 0.8424,
 'f1': 0.8718691782239701,
 'roc_auc': 0.95135584}

In [44]:
prob_cnn = cnn_model.predict(X_test_pad).ravel()
pred_cnn = (prob_cnn >= 0.5).astype(int)

metrics_cnn = evaluate_predictions(y_test, pred_cnn, prob_cnn)

print("1D CNN")
print(metrics_cnn)
print("\nClassification report:\n")
print(classification_report(y_test, pred_cnn, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, pred_cnn))

model_dir = "assignment2/cnn_1d"
os.makedirs(model_dir, exist_ok=True)

cnn_model.save(os.path.join(model_dir, "model.keras"))

readme_cnn = """# 1D CNN for IMDB Sentiment Classification

## Model architecture
- Input: tokenized and padded review sequences
- Embedding layer (trainable)
- Conv1D with 128 filters and kernel size 5
- GlobalMaxPooling1D
- Dense layer with ReLU
- Dropout
- Output sigmoid layer

## Techniques applied
- text cleaning
- tokenization and padding
- 1D convolution for local phrase patterns
- dropout regularization
- early stopping
- evaluation with accuracy, precision, recall, F1, and ROC-AUC
"""

save_results_and_readme(model_dir, metrics_cnn, readme_cnn)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
1D CNN
{'accuracy': 0.8762, 'precision': 0.9034749034749034, 'recall': 0.8424, 'f1': 0.8718691782239701, 'roc_auc': 0.95135584}

Classification report:

              precision    recall  f1-score   support

           0     0.8524    0.9100    0.8802      5000
           1     0.9035    0.8424    0.8719      5000

    accuracy                         0.8762     10000
   macro avg     0.8779    0.8762    0.8761     10000
weighted avg     0.8779    0.8762    0.8761     10000

Confusion matrix:
 [[4550  450]
 [ 788 4212]]


In [45]:
comparison = pd.DataFrame([
    {"model": "Tuned MLP (lexicon)", **metrics_lex},
    {"model": "LSTM", **metrics_lstm},
    {"model": "1D CNN", **metrics_cnn},
])

comparison = comparison.sort_values("accuracy", ascending=False).reset_index(drop=True)
comparison

,model,accuracy,precision,recall,f1,roc_auc
0,1D CNN,0.8762,0.903475,0.8424,0.871869,0.951356
1,LSTM,0.7887,0.781768,0.8010,0.791267,0.834293
2,Tuned MLP (lexicon),0.7740,0.774219,0.7736,0.773910,0.859655


In [46]:
comparison.to_csv("assignment2/model_comparison.csv", index=False)

comparison_md = """# Assignment 2 Model Comparison

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC |
|------|----------|-----------|--------|----|---------|
"""

for _, row in comparison.iterrows():
    comparison_md += f"| {row['model']} | {row['accuracy']:.4f} | {row['precision']:.4f} | {row['recall']:.4f} | {row['f1']:.4f} | {row['roc_auc']:.4f} |\n"

with open("assignment2/comparison.md", "w") as f:
    f.write(comparison_md)

comparison

,model,accuracy,precision,recall,f1,roc_auc
0,1D CNN,0.8762,0.903475,0.8424,0.871869,0.951356
1,LSTM,0.7887,0.781768,0.8010,0.791267,0.834293
2,Tuned MLP (lexicon),0.7740,0.774219,0.7736,0.773910,0.859655
